# 量子神经网络(QNN)用于H2分子VMC计算

本notebook实现了一个量子神经网络(QNN)模型，用于H2分子的变分蒙特卡洛(VMC)计算。

主要特点：
- 量子电路支持批量输入
- 修复了PRNGKey序列化问题
- 统一使用float32类型避免类型不匹配
- 经典-量子混合神经网络架构

In [1]:
import jax
import jax.numpy as jnp
import pennylane as qml
from flax import nnx
from functools import partial
import numpy as np
import netket as nk
import netket.experimental as nkx
import matplotlib.pyplot as plt

# 禁用JAX的x64模式，使用float32
jax.config.update("jax_enable_x64", False)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def quantum_neural_network(x, params, n_qubits, n_layers):
    """
    兼容Batch的量子电路核心逻辑：
    - x: 输入张量，形状为 (batch_size, n_qubits)
    - params: 量子电路参数，形状为 (n_layers, 2 * n_qubits)
    - 所有量子门操作自动沿Batch维度向量化
    """
    # 确保输入是二维张量
    x = jnp.atleast_2d(x)
    
    # 校验特征维度
    if x.shape[-1] != n_qubits:
        raise ValueError(f"输入特征维度需为{n_qubits}，当前为{x.shape[-1]}")
    
    # 数据编码：向量化RX门
    for i in range(n_qubits):
        qml.RX(x[:, i] * jnp.pi, wires=i)
    
    # 变分层：向量化旋转/纠缠门
    for layer in range(n_layers):
        # 纠缠层
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
        qml.Barrier(wires=range(n_qubits))
        
        # 旋转层
        for i in range(n_qubits):
            qml.RX(params[layer, 2*i], wires=i)
            qml.RZ(params[layer, 2*i+1], wires=i)
    
    # 测量：返回每个量子比特的期望值
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [3]:
def qnn_circuit(n_qubits, n_layers):
    """创建量子电路函数"""
    dev = qml.device('default.qubit', wires=n_qubits)
    qnode = qml.QNode(
        func=quantum_neural_network, 
        device=dev, 
        interface='jax'
    )
    return partial(qnode, n_qubits=n_qubits, n_layers=n_layers)

In [29]:
class SimpleQNNModel(nnx.Module):
    """简化的QNN模型，完全避免PRNGKey存储问题"""
    
    def __init__(self, n_qubits: int, n_layers: int, rngs: nnx.Rngs):
        # 直接初始化参数，不存储rngs
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        
        # 初始化QNN参数
        qnn_key = rngs.params()
        self.qnn_params = nnx.Param(
            jax.random.normal(qnn_key, (n_layers, 2*n_qubits), dtype=jnp.float32)
        )
        
        self.linear = nnx.Linear(in_features=n_qubits, out_features=n_qubits, rngs=rngs,use_bias=False)
        
        # 创建QNN函数
        self.qnn_func = qnn_circuit(n_qubits=n_qubits, n_layers=n_layers)
    
    def __call__(self, x):
        # 确保输入是float32类型
        x = jnp.asarray(x, dtype=jnp.float32)
        
        # 获取QNN输出
        qnn_output = self.qnn_func(x=x, params=self.qnn_params)
        qnn_output = jnp.asarray(qnn_output, dtype=jnp.float32).reshape(-1,self.n_qubits)
        
        # 如果是单个样本，确保形状正确
        if qnn_output.ndim == 1:
            qnn_output = qnn_output.reshape(1, -1)
        
        # 线性变换
        y = self.linear(qnn_output)
        y = nnx.relu(y)
        
        # 返回标量输出
        return jnp.sum(y, axis=-1)

In [30]:
rngs = nnx.Rngs(params=0)
key = rngs.params()
qnn_func = qnn_circuit(n_qubits=4, n_layers=2)
y = qnn_func(x=[[1,0,1,0],[0,1,0,1]],params= nnx.Param(jax.random.normal(key, (2,4))))
jnp.array(y).reshape(-1,4)

Array([[-0.09625137,  0.09625134,  0.3746891 , -0.21497852],
       [-0.02295583, -0.02295598, -0.01601771, -0.01601768]],      dtype=float32)

In [31]:
model = SimpleQNNModel(rngs=nnx.Rngs(params=0),n_qubits=4, n_layers=2)
model(x = jnp.asarray([[1,0,1,0],[0,1,0,1]], dtype=jnp.float32))

Array([0.5608727, 0.       ], dtype=float32)

## H2分子系统设置

In [32]:
from pyscf import gto, scf, fci

# 设置H2分子的几何构型
bond_length = 0.74  # H2平衡键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

# 创建分子对象，使用STO-3G基组
mol = gto.M(atom=geometry, basis='STO-3G')

# 进行Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"Hartree-Fock能量: {E_hf:.8f} Ha")

# 进行FCI计算作为参考
cisolver = fci.FCI(mf)
E_fci, fcivec = cisolver.kernel()
print(f"FCI能量: {E_fci:.8f} Ha")

# 使用NetKet创建哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

Hartree-Fock能量: -1.11675931 Ha
FCI能量: -1.13728383 Ha


In [33]:
# 定义希尔伯特空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,  # 总空间轨道数
    s=1/2,
    n_fermions_per_spin=(1, 1)  # 每种自旋的电子数
)

# 创建图结构
g = nk.graph.Graph(edges=[(0,1),(2,3)])

# 创建采样器
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)

## 创建QNN模型

In [34]:
# 创建QNN模型
model = SimpleQNNModel(n_qubits=4, n_layers=2, rngs=nnx.Rngs(42))

# 测试模型
test_input = jnp.array([[1,0,1,0]])
output = model(test_input)
print(f"模型测试输出: {output}")

模型测试输出: [0.23016946]


## VMC计算设置

In [35]:
# 创建变分状态
vstate = nk.vqs.MCState(sampler, model, n_samples=1008)
print(f"变分状态参数数量: {vstate.n_parameters}")

变分状态参数数量: 32


In [47]:
# 设置优化器
opt = nk.optimizer.Sgd(learning_rate=0.05)
sr = nk.optimizer.SR(diag_shift=0.01)

# 创建VMC驱动器
gs = nk.driver.VMC(ha, opt, variational_state=vstate, preconditioner=sr)

# 添加日志记录器
exp_name = "h2_molecule_qnn"
gs.run(n_iter=30, out=exp_name)

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [00:09<00:00,  3.03it/s, Energy=nan ± nan [σ²=nan]]


(JsonLog('h2_molecule_qnn', mode=write, autoflush_cost=0.005)
   Runtime cost:
   	Log:    0.01923513412475586
   	Params: 0.014458417892456055,)

In [50]:
with open(f"{exp_name}.log") as f:
    data = json.load(f)

In [54]:
type(data)

dict

In [58]:
data

{'acceptance': {'iters': [0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29],
  'value': [0.0279182,
   0.028435202,
   0.029813878,
   0.029411765,
   0.023466222,
   0.02460076,
   0.03219784,
   0.02699908,
   0.029095817,
   0.031364888,
   0.027042165,
   0.027501723,
   0.02437098,
   0.023609834,
   0.030101104,
   0.028708065,
   0.02618049,
   0.02780331,
   0.028191062,
   0.027386833,
   0.026381548,
   0.027544808,
   0.027444279,
   0.028722426,
   0.027602252,
   0.028349034,
   0.028621897,
   0.026568245,
   0.025907628,
   0.029325597]},
 'Energy': {'iters': [0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29],
  'Mean': [None,
   None,
   None,
   None,
   None,
   None,
   None,
  

## 结果分析

In [49]:
# 读取结果
import json

# 读取日志数据
with open(f"{exp_name}.log") as f:
    data = json.load(f)

x = data["Energy"]["iters"]
y = data["Energy"]["Mean"]['real']

# 绘制能量收敛曲线
plt.figure(figsize=(10, 6))
plt.axhline(ed_energies[0], color="red", linestyle="--", label=f"FCI energy = {E_fci:.6f} Ha")
plt.plot(x, y, 'b-', label="VMC energy")
plt.xlabel("Iterations")
plt.ylabel("Energy (Ha)")
plt.title("H2 molecule energy convergence")
plt.legend()
plt.grid(True)
plt.show()

# 打印最终结果
print(f"\n最终VMC能量: {y[-1]:.8f} Ha")
print(f"与FCI能量误差: {abs(y[-1] - E_fci):.8f} Ha")


TypeError: list indices must be integers or slices, not str